In [4]:
# openalex_search.py  (fixed year filter)
import time, os, requests, json
from typing import Optional, Callable, Dict, Any, List
from enhanced_query_script import wait_for_rate_limit

try:
    from enhanced_query_script import norm_openalex as _norm_oa
except ImportError:
    _norm_oa = None

include_preprints = True


ALLOWED_OA_TYPES = ["article", "review"]
if include_preprints:
    ALLOWED_OA_TYPES.append("preprint")

NEGATIVE_TOKENS = ("human", "clinical", "patient", "medical", "veterinary")

def _fallback_norm_openalex(item: Dict[str, Any]) -> Dict[str, Any]:
    def _abstract_from_inv(inv):
        if not inv: return None
        maxpos = max((max(v) for v in inv.values() if v), default=-1)
        words = [None] * (maxpos + 1)
        for w, poss in inv.items():
            for p in poss:
                if 0 <= p < len(words) and words[p] is None:
                    words[p] = w
        return " ".join([w for w in words if w])
    abs_txt = _abstract_from_inv(item.get("abstract_inverted_index"))
    authors = []
    for a in (item.get("authorships") or []):
        name = a.get("author", {}).get("display_name")
        if name: authors.append(name)
    host = item.get("host_venue") or {}
    primary = item.get("primary_location") or {}
    venue = host.get("display_name") or (primary.get("source") or {}).get("display_name")
    return {
        "id": item.get("id"),
        "doi": item.get("doi"),
        "title": item.get("display_name"),
        "year": item.get("publication_year"),
        "authors": authors,
        "venue": venue,
        "url": primary.get("landing_page_url") or primary.get("pdf_url") or host.get("url"),
        "type": item.get("type"),
        "cited_by_count": item.get("cited_by_count"),
        "abstract": abs_txt,
        "open_access": (item.get("open_access") or {}).get("is_oa"),
    }

def search_openalex(
    query: str,
    on_record: Optional[Callable[[Dict[str, Any]], None]] = None,
    year_min: int = 2018,
    year_max: Optional[int] = None,
    mailto: Optional[str] = None,
    language: str = "en",
    include_preprints: bool = True,
    has_abstract: Optional[bool] = None,
    sort: str = "cited_by_count:desc",    # or "relevance_score:desc"
    per_page: int = 200,
    client_side_negatives: bool = True,
    verbose_errors: bool = True,
) -> List[Dict[str, Any]]:
    base_url = "https://api.openalex.org/works"
    headers = {"Accept": "application/json", "User-Agent": "AgriReviewBot/1.0 (OpenAlex search)"}

    types = ["article", "review"]          # the modern OA types you want
    if include_preprints:
        types.append("preprint")

    # ✅ Use date-range filters instead of publication_year:>=
    filters = [
        f"from_publication_date:{year_min}-01-01",
        f"language:{language}",
        f"type:{'|'.join(types)}",
        "is_paratext:false"
    ]
    if year_max:
        filters.append(f"to_publication_date:{year_max}-12-31")
    if has_abstract is True:
        filters.append("has_abstract:true")
    elif has_abstract is False:
        filters.append("has_abstract:false")

    SELECT_SAFE = ",".join([
        "id","doi","display_name","publication_year","type",
        "abstract_inverted_index","authorships","primary_location",
        "cited_by_count","open_access"
    ])  # note: host_venue removed per API error

    params = {
        "search": query,
        "filter": ",".join(filters),
        "per-page": str(per_page),
        "cursor": "*",
        "select": SELECT_SAFE,
        "sort": sort,
    }
    if mailto:
        params["mailto"] = mailto

    out: List[Dict[str, Any]] = []
    session = requests.Session()

    def _request_with_fallback(p: dict) -> requests.Response:
        for attempt in range(3):
            wait_for_rate_limit(1.0)
            # print the full url for debugging
            print(f"[OpenAlex Request] {base_url}?{'&'.join(f'{k}={v}' for k,v in p.items())}")
            resp = session.get(base_url, params=p, headers=headers, timeout=30)
            if resp.status_code in (429, 500, 502, 503):
                ra = resp.headers.get("Retry-After")
                time.sleep(float(ra) if (ra and ra.isdigit()) else 1.5)
                continue
            if resp.status_code == 400:
                if attempt == 0 and "select" in p:
                    if verbose_errors:
                        try: print(f"[OpenAlex 400] Dropping select; server said:", resp.json())
                        except Exception: print(f"[OpenAlex 400] Dropping select; raw:", resp.text)
                    p = {**p}; p.pop("select", None)
                    continue
                if attempt == 1 and "sort" in p:
                    if verbose_errors:
                        try: print(f"[OpenAlex 400] Dropping sort; server said:", resp.json())
                        except Exception: print(f"[OpenAlex 400] Dropping sort; raw:", resp.text)
                    p = {**p}; p.pop("sort", None)
                    continue
            resp.raise_for_status()
            return resp
        resp.raise_for_status()
        return resp

    while True:
        resp = _request_with_fallback(dict(params))
        payload = resp.json()

        for item in payload.get("results", []):
            t = (item.get("type") or "").lower()
            if t and t not in ALLOWED_OA_TYPES:
                continue
            rec = _norm_oa(item) if _norm_oa else _fallback_norm_openalex(item)
            if rec and (not rec.get("year") or rec["year"] >= year_min):
                if client_side_negatives:
                    txt = f"{(rec.get('title') or '').lower()} {(rec.get('abstract') or '').lower()}"
                    if any(neg in txt for neg in NEGATIVE_TOKENS):
                        continue
                out.append(rec)
                if on_record: on_record(rec)

        nxt = (payload.get("meta") or {}).get("next_cursor")
        if not nxt: break
        params["cursor"] = nxt

    return out


In [5]:
# run_openalex.py
import os, json, datetime
from enhanced_query_script import to_jsonl, to_csv

from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = os.path.join("../outputs", f"openalex_run_{timestamp}")
os.makedirs(outdir, exist_ok=True)

queries_file = "../queries.json"
# queries_file = "../queries/queries_openalex.json"
with open(queries_file, "r", encoding="utf-8") as f:
    queries = json.load(f)

for idx, q in enumerate(queries, start=1):
    print(f"\n=== OpenAlex Query {idx}/{len(queries)} ===")
    print(q)
    results = search_openalex(
        q,
        year_min=2018,
        mailto=os.getenv("CROSSREF_EMAIL"),      # set MAILTO in env for rate-limit friendliness
        language="en",
        include_preprints=True,
        has_abstract=None,               # set True to require abstracts
        sort="cited_by_count:desc",      # or "relevance_score:desc"
        per_page=200,
        client_side_negatives=True
    )
    for r in results:
        r["query"]   = q
        r["query_id"]= f"Q{idx:02d}"
    to_jsonl(os.path.join(outdir, f"Q{idx:02d}_results.jsonl"), results)
    to_csv  (os.path.join(outdir, f"Q{idx:02d}_results.csv"),   results)
    print(f"Found {len(results)} results")



=== OpenAlex Query 1/32 ===
"unsupervised domain adaptation" AND ("plant disease" OR pest) AND (plant OR crop OR agriculture)
[OpenAlex Request] https://api.openalex.org/works?search="unsupervised domain adaptation" AND ("plant disease" OR pest) AND (plant OR crop OR agriculture)&filter=from_publication_date:2018-01-01,language:en,type:article|review|preprint,is_paratext:false&per-page=200&cursor=*&select=id,doi,display_name,publication_year,type,abstract_inverted_index,authorships,primary_location,cited_by_count,open_access&sort=cited_by_count:desc&mailto=bekhouche.mouadh@univ-oeb.dz
[OpenAlex Request] https://api.openalex.org/works?search="unsupervised domain adaptation" AND ("plant disease" OR pest) AND (plant OR crop OR agriculture)&filter=from_publication_date:2018-01-01,language:en,type:article|review|preprint,is_paratext:false&per-page=200&cursor=IlswLCAnLUluZmluaXR5JywgMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1czMTI5NTE1MTA1J10i&select=id,doi,display_name,publication_year,type,abstrac

In [3]:
emailto = os.getenv("CROSSREF_EMAIL")
print(emailto)

bekhouche.mouadh@univ-oeb.dz


In [6]:
# aggregate all results
import glob
all_results = []
for fn in glob.glob(os.path.join(outdir, "Q??_results.jsonl")):
    with open(fn, "r", encoding="utf-8") as f:
        for line in f:
            all_results.append(json.loads(line))
print(f"Total aggregated results: {len(all_results)}")
to_jsonl(os.path.join(outdir, f"all_results.jsonl"), all_results)
to_csv  (os.path.join(outdir, f"all_results.csv"),   all_results)

Total aggregated results: 2503
